In [0]:
%pip install databricks_vectorsearch 
dbutils.library.restartPython()

In [0]:
from config import DeployConfig
from databricks.vector_search.client import *
from tqdm import tqdm

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
image_table = 'ml_shovakeemian.feip.pet_images'
image_embeddings= 'ml_shovakeemian.feip.pet_images_embeddings'
vs_index_std = 'ml_shovakeemian.vs_benchmarking.pet_images_vs_standard'
vs_index_stg = 'ml_shovakeemian.vs_benchmarking.pet_images_vs_storage_op'
vs_index_delta_sync = 'ml_shovakeemian.vs_benchmarking.pet_images_delta_sync'

In [0]:
vs_index = getattr(cfg, f"vs_index")
image_table = getattr(cfg, f"image_table")

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [0]:
w.secrets.list_secrets(scope='shovakeemian-scope')

In [0]:
vsc = VectorSearchClient(
    workspace_url="https://e2-demo-field-eng.cloud.databricks.com/",
    service_principal_client_id=dbutils.secrets.get(scope="shovakeemian-scope", key="shovakeemian-sp-client-id"),
    service_principal_client_secret=dbutils.secrets.get(scope="shovakeemian-scope", key="shovakeemian-sp-client-secret")
)

In [0]:
# client = VectorSearchClient(
#     workspace_url="https://e2-demo-field-eng.cloud.databricks.com/",
#     service_principal_client_id="e9dfc794-21ab-4ccb-b4a0-1ec3725bc501",
#     service_principal_client_secret="dosefd624858d2b198049ce9bf04def159ee"
# )

In [0]:
# vsc = VectorSearchClient()

In [0]:
spark.sql(f'ALTER TABLE {vs_index.delta_sync_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')

In [0]:
vs_index.endpoint

In [0]:
vs_index.path

In [0]:
vs_index_std

In [0]:
vsc.create_delta_sync_index(
    endpoint_name='one-env-shared-endpoint-12',
    index_name=vs_index.path,
    source_table_name=vs_index.delta_sync_table,
    columns_to_sync=['id', 'path', 'model_input'],
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_dimension=768,
    embedding_vector_column="image_embeddings"
)

In [0]:
#delta sync tables
spark.sql(f'''
          create table if not exists {vs_index_delta_sync} as 
          select 
            a.id,
            a.path,
            a.content as image_byte_array,
            a.model_input as base64str_image,
            b.image_embeddings
          from {image_table} a 
          join {image_embeddings} b on a.id=b.id
          ''')

In [0]:
display(spark.sql(f'select * from {vs_index_delta_sync} limit 10;')) 

In [0]:
spark.sql(f'ALTER TABLE {vs_index_delta_sync} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)')

In [0]:
benchmarking_dataset = [
'Abyssinian cat',
'African Grey Parrot',
'American Shorthair cat',
'Beagle dog',
"Bourke's Parakeet",
'Boxer dog',
'British Shorthair cat',
'Bulldog dog',
'Burmese cat',
'Cocker Spaniel dog',
'Dachshund dog',
'French Bulldog dog',
'German Shepherd dog',
'Golden Retriever dog',
'Great Dane dog',
'Guinea pig',
'Hamster',
'Labrador Retriever dog',
'Lovebird',
'Maine Coon cat',
'Miniature Schnauzer dog',
'Parrotlet',
'Persian cat',
'Pionus Parrot',
'Poodle dog',
'Quaker Parrot',
'Rabbit',
'Ragdoll cat',
'Rottweiler dog',
'Russian Blue cat',
'Scottish Fold cat',
'Shih Tzu dog',
'Siamese cat',
'Yorkshire Terrier dog'
]


In [0]:
import mlflow
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from io import BytesIO

In [0]:
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

In [0]:
def get_text_embedding(text):
    inputs = processor(text=text, return_tensors="pt", padding=True)
    text_features = model.get_text_features(**inputs)
    return text_features.detach().numpy().tolist()[0]

In [0]:
text="black forest cat"
query=get_text_embedding(text)

In [0]:
len(query)

In [0]:
query

# STANDARD VS

In [0]:
client = VectorSearchClient()

In [0]:
vs_index_delta_sync

In [0]:
vs_index_std

In [0]:
# making index (does not allow metadata columns of byte array)
client.create_delta_sync_index(
    endpoint_name='one-env-shared-endpoint-11',
    index_name=vs_index_std,
    source_table_name=vs_index_delta_sync,
    columns_to_sync=['path', 'base64str_image'],
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_dimension=768,
    embedding_vector_column="image_embeddings"
)

In [0]:
query_embedding

In [0]:
index = vsc.get_index(endpoint_name='one-env-shared-endpoint-11', index_name=vs_index_std)


query_embedding=get_text_embedding(benchmarking_dataset[0])

vs_output = index.similarity_search(
            query_vector=query_embedding,
            columns=['id', 'path', 'base64str_image'],
            num_results=1,
            debug_level=1
            )

In [0]:
rows = (vs_output or {}).get("result", {}).get("data_array", []) or []

In [0]:
vs_output

In [0]:
#benchmarking

index = vsc.get_index(endpoint_name='one-env-shared-endpoint-11', index_name=vs_index_std)

all_debug = []
for query_text in tqdm(benchmarking_dataset):
    query_embedding=get_text_embedding(query_text)
    vs_output = index.similarity_search(
                query_vector=query_embedding,
                columns=['id', 'path', 'base64str_image'],
                num_results=1,
                debug_level=1
                )
    rows = (vs_output or {}).get("result", {}).get("data_array", []) or []
    paths = []
    for r in rows[:limit]:
      p = (r[2] or "").replace("dbfs:/", "/")
      if p:
        paths.append(p)
    w.files.download(path).contents.read()
    end = time.time()
    debug_info = resp["debug_info"]
    debug_info["total_elapsed"] = (end - start) * 1000
    all_debug.append(debug_info)

# STORAGE OPTIMIZED VS

In [0]:
vs_index_stg

In [0]:
# making index (does not allow metadata columns of byte array)
client.create_delta_sync_index(
    endpoint_name='shovakeemian_vs_endpoint_stg',
    index_name=vs_index_stg,
    source_table_name=vs_index_delta_sync,
    columns_to_sync=['id', 'path', 'base64str_image'],
    pipeline_type="TRIGGERED",
    primary_key="id",
    embedding_dimension=768,
    embedding_vector_column="image_embeddings"
)